<a href="https://colab.research.google.com/github/hamzafahim691-debug/Decodelab.project/blob/main/Projecttt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(88)
n_samples = 1200

ad_spend = np.random.gamma(shape=3, scale=15, size=n_samples)
impressions = ad_spend * 12.4 + np.random.normal(0, 8, n_samples)
impressions = np.clip(impressions, 100, 5000)
clicks = np.random.normal(loc=250, scale=35, size=n_samples)

anomaly_idx = np.random.choice(n_samples, size=40, replace=False)
clicks[anomaly_idx[:20]] = clicks[anomaly_idx[:20]] * 6
clicks[anomaly_idx[20:]] = -15

channels = np.random.choice(['Social_Media', 'Search_Ads', 'Email_Promo'], size=n_samples, p=[0.5, 0.3, 0.2])
revenue_generated = (ad_spend * 8) + (clicks * 0.25) + np.random.normal(0, 15, n_samples)

mask_spend = np.random.rand(n_samples) < 0.07
mask_clicks = np.random.rand(n_samples) < 0.23
mask_channel = np.random.rand(n_samples) < 0.04

ad_spend[mask_spend] = np.nan
clicks[mask_clicks] = np.nan
channels = np.where(mask_channel, None, channels)

df_marketing = pd.DataFrame({
    'Ad_Spend_USD': ad_spend,
    'Total_Impressions': impressions,
    'Link_Clicks': clicks,
    'Marketing_Channel': channels,
    'Revenue_Generated': revenue_generated
})
print(df_marketing.shape)

(1200, 5)


In [ ]:
from sklearn.impute import KNNImputer

df_marketing = df_marketing.dropna(subset=['Marketing_Channel'])
median_spend = df_marketing['Ad_Spend_USD'].median()
df_marketing['Ad_Spend_USD'] = df_marketing['Ad_Spend_USD'].fillna(median_spend)

knn_engine = KNNImputer(n_neighbors=5)
numeric_cols = ['Ad_Spend_USD', 'Total_Impressions', 'Link_Clicks']
df_marketing[numeric_cols] = knn_engine.fit_transform(df_marketing[numeric_cols])

Q1_m = df_marketing['Link_Clicks'].quantile(0.25)
Q3_m = df_marketing['Link_Clicks'].quantile(0.75)
IQR_m = Q3_m - Q1_m
df_marketing['Link_Clicks'] = np.clip(df_marketing['Link_Clicks'], Q1_m - 1.5 * IQR_m, Q3_m + 1.5 * IQR_m)

df_marketing['Spend_Per_Click'] = df_marketing['Ad_Spend_USD'] / (df_marketing['Link_Clicks'] + 1.0)
df_marketing['Click_Through_Ratio'] = df_marketing['Link_Clicks'] * df_marketing['Total_Impressions']
df_marketing['Spend_Deviation'] = df_marketing['Ad_Spend_USD'] - df_marketing['Ad_Spend_USD'].mean()

df_final_encoded = pd.get_dummies(df_marketing, columns=['Marketing_Channel'], drop_first=False, dtype=int)
df_marketing_clean = df_final_encoded.drop(columns=['Total_Impressions'])

df_marketing_clean.to_csv('marketing_campaign_clean_production.csv', index=False)
df_marketing_clean.head()

,Ad_Spend_USD,Link_Clicks,Revenue_Generated,Spend_Per_Click,Click_Through_Ratio,Spend_Deviation,Marketing_Channel_Email_Promo,Marketing_Channel_Search_Ads,Marketing_Channel_Social_Media
0,42.675657,227.679005,397.594071,0.186618,120690.593322,-2.826971,0,0,1
1,122.010648,242.969736,1031.837298,0.500106,370461.006584,76.508020,0,1,0
2,72.296779,209.062096,638.877228,0.344169,187137.026287,26.794151,0,0,1
3,69.735092,247.255732,622.224025,0.280900,212743.032763,24.232464,0,1,0
4,35.951001,263.233372,355.746340,0.136058,114565.600355,-9.551627,0,1,0


In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

np.random.seed(88)
n_samples = 1200

ad_spend = np.random.gamma(shape=3, scale=15, size=n_samples)
impressions = ad_spend * 12.4 + np.random.normal(0, 8, n_samples)
impressions = np.clip(impressions, 100, 5000)
clicks = np.random.normal(loc=250, scale=35, size=n_samples)

anomaly_idx = np.random.choice(n_samples, size=40, replace=False)
clicks[anomaly_idx[:20]] = clicks[anomaly_idx[:20]] * 6
clicks[anomaly_idx[20:]] = -15

channels = np.random.choice(['Social_Media', 'Search_Ads', 'Email_Promo'], size=n_samples, p=[0.5, 0.3, 0.2])
revenue_generated = (ad_spend * 8) + (clicks * 0.25) + np.random.normal(0, 15, n_samples)

mask_spend = np.random.rand(n_samples) < 0.07
mask_clicks = np.random.rand(n_samples) < 0.23
mask_channel = np.random.rand(n_samples) < 0.04

ad_spend[mask_spend] = np.nan
clicks[mask_clicks] = np.nan
channels = np.where(mask_channel, None, channels)

df_marketing = pd.DataFrame({
    'Ad_Spend_USD': ad_spend,
    'Total_Impressions': impressions,
    'Link_Clicks': clicks,
    'Marketing_Channel': channels,
    'Revenue_Generated': revenue_generated
})

df_marketing = df_marketing.dropna(subset=['Marketing_Channel'])
median_spend = df_marketing['Ad_Spend_USD'].median()
df_marketing['Ad_Spend_USD'] = df_marketing['Ad_Spend_USD'].fillna(median_spend)

knn_engine = KNNImputer(n_neighbors=5)
numeric_cols = ['Ad_Spend_USD', 'Total_Impressions', 'Link_Clicks']
df_marketing[numeric_cols] = knn_engine.fit_transform(df_marketing[numeric_cols])

Q1_m = df_marketing['Link_Clicks'].quantile(0.25)
Q3_m = df_marketing['Link_Clicks'].quantile(0.75)
IQR_m = Q3_m - Q1_m
df_marketing['Link_Clicks'] = np.clip(df_marketing['Link_Clicks'], Q1_m - 1.5 * IQR_m, Q3_m + 1.5 * IQR_m)

df_marketing['Spend_Per_Click'] = df_marketing['Ad_Spend_USD'] / (df_marketing['Link_Clicks'] + 1.0)
df_marketing['Click_Through_Ratio'] = df_marketing['Link_Clicks'] * df_marketing['Total_Impressions']
df_marketing['Spend_Deviation'] = df_marketing['Ad_Spend_USD'] - df_marketing['Ad_Spend_USD'].mean()

df_final_encoded = pd.get_dummies(df_marketing, columns=['Marketing_Channel'], drop_first=False, dtype=int)
df_marketing_clean = df_final_encoded.drop(columns=['Total_Impressions'])

df_marketing_clean.to_csv('marketing_campaign_clean_production.csv', index=False)

ipynb_structure = {
 "cells": [
  {"cell_type": "markdown", "metadata": {}, "source": ["# DecodeLabs Project 1: Advanced EDA & Feature Engineering Pipeline\\n", "### Domain Matrix Variant: Digital Marketing Analytics\\n"]},
  {"cell_type": "markdown", "metadata": {}, "source": ["## Step 1: Input Infrastructure (Generating Raw Matrix)"]},
  {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
    "import pandas as pd\\nimport numpy as np\\n\\nnp.random.seed(88)\\nn_samples = 1200\\n\\nad_spend = np.random.gamma(shape=3, scale=15, size=n_samples)\\nimpressions = ad_spend * 12.4 + np.random.normal(0, 8, n_samples)\\nimpressions = np.clip(impressions, 100, 5000)\\nclicks = np.random.normal(loc=250, scale=35, size=n_samples)\\n\\nanomaly_idx = np.random.choice(n_samples, size=40, replace=False)\\nclicks[anomaly_idx[:20]] = clicks[anomaly_idx[:20]] * 6\\nclicks[anomaly_idx[20:]] = -15\\n\\nchannels = np.random.choice(['Social_Media', 'Search_Ads', 'Email_Promo'], size=n_samples, p=[0.5, 0.3, 0.2])\\nrevenue_generated = (ad_spend * 8) + (clicks * 0.25) + np.random.normal(0, 15, n_samples)\\n\\nmask_spend = np.random.rand(n_samples) < 0.07\\nmask_clicks = np.random.rand(n_samples) < 0.23\\nmask_channel = np.random.rand(n_samples) < 0.04\\n\\nad_spend[mask_spend] = np.nan\\nclicks[mask_clicks] = np.nan\\nchannels = np.where(mask_channel, None, channels)\\n\\ndf_marketing = pd.DataFrame({\\n    'Ad_Spend_USD': ad_spend,\\n    'Total_Impressions': impressions,\\n    'Link_Clicks': clicks,\\n    'Marketing_Channel': channels,\\n    'Revenue_Generated': revenue_generated\\n})\\nprint(df_marketing.shape)"
  ]},
  {"cell_type": "markdown", "metadata": {}, "source": ["## Step 2: Process Stage (Part A) - The Missing Data Decision Matrix"]},
  {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
    "from sklearn.impute import KNNImputer\\n\\ndf_marketing = df_marketing.dropna(subset=['Marketing_Channel'])\\nmedian_spend = df_marketing['Ad_Spend_USD'].median()\\ndf_marketing['Ad_Spend_USD'] = df_marketing['Ad_Spend_USD'].fillna(median_spend)\\n\\nknn_engine = KNNImputer(n_neighbors=5)\\nnumeric_cols = ['Ad_Spend_USD', 'Total_Impressions', 'Link_Clicks']\\ndf_marketing[numeric_cols] = knn_engine.fit_transform(df_marketing[numeric_cols])\\nprint(df_marketing.isnull().sum())"
  ]},
  {"cell_type": "markdown", "metadata": {}, "source": ["## Step 3: Process Stage (Part B) - Outlier Isolation & Winsorization"]},
  {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
    "Q1_m = df_marketing['Link_Clicks'].quantile(0.25)\\nQ3_m = df_marketing['Link_Clicks'].quantile(0.75)\\nIQR_m = Q3_m - Q1_m\\ndf_marketing['Link_Clicks'] = np.clip(df_marketing['Link_Clicks'], Q1_m - 1.5 * IQR_m, Q3_m + 1.5 * IQR_m)"
  ]},
  {"cell_type": "markdown", "metadata": {}, "source": ["## Step 4: Process Stage (Part C) - Vectorized Feature Engineering"]},
  {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
    "df_marketing['Spend_Per_Click'] = df_marketing['Ad_Spend_USD'] / (df_marketing['Link_Clicks'] + 1.0)\\ndf_marketing['Click_Through_Ratio'] = df_marketing['Link_Clicks'] * df_marketing['Total_Impressions']\\ndf_marketing['Spend_Deviation'] = df_marketing['Ad_Spend_USD'] - df_marketing['Ad_Spend_USD'].mean()"
  ]},
  {"cell_type": "markdown", "metadata": {}, "source": ["## Step 5: Output Stage - Structural Consolidation & Multicollinearity Eradication"]},
  {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
    "df_final_encoded = pd.get_dummies(df_marketing, columns=['Marketing_Channel'], drop_first=False, dtype=int)\\ndf_marketing_clean = df_final_encoded.drop(columns=['Total_Impressions'])\\ndf_marketing_clean.to_csv('marketing_campaign_clean_production.csv', index=False)\\ndf_marketing_clean.head()"
  ]}
 ],
 "metadata": {"language_info": {"name": "python"}},
 "nbformat": 4, "nbformat_minor": 2
}

with open('DecodeLabs_Project1_Marketing_Variant.ipynb', 'w') as f:
    json.dump(ipynb_structure, f, indent=1)

In [ ]:
from google.colab import files
files.download('marketing_campaign_clean_production.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('DecodeLabs_Project1_Marketing_Variant.ipynb')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>